Notebook for exploring creating this project's Extract function. Don't forget to set the Kernel to this project's .venv

This is code for initial load of 2026 stock price data

In [ ]:
import json
from datetime import UTC, datetime

import boto3
import requests
from config import settings

s3 = boto3.client('s3')
api_url = 'https://financialmodelingprep.com/stable/historical-price-eod/full?symbol='
timeout = (3.05, 30)
filename = "stock_prices_raw_" + datetime.now(UTC).strftime("%Y%m%d_%H%M%S") + ".json"
bucket = 'phile-findata1-raw-data'
key_path = 'to_process/'

def extract_eod_stockprices(api_url: str, timeout:tuple) -> list[dict]:
    all_records = []
    config_bucket = 'phile-findata1-configdata'
    key_file = 'fmp87.json'

    config_response = s3.get_object(Bucket=config_bucket, Key=key_file)
    config_content = config_response['Body'].read().decode('utf-8')
    json_config = json.loads(config_content)

    for ticker in json_config['fmp87']:
        response = requests.get(
            api_url + ticker,
            params={'from': '2026-01-01', 'to': '2026-09-04', 'apikey': settings.apikey},
            timeout=timeout
        )
        response.raise_for_status()
        data = response.json()
        all_records.extend(data)
       
    return all_records

stock_prices = extract_eod_stockprices(api_url, timeout)

s3.put_object(
    Bucket=bucket,
    Key=key_path + filename,
    Body=json.dumps(stock_prices),
    ContentType='application/json'
)


{'ResponseMetadata': {'RequestId': '3QNBEXEA1NT4MQ5J',
  'HostId': '6+OMon/6TFfsFnafECM70JJOupE62RQmsFe4fVlZ4svmZtt/QoYbcTVXcCiDkuWyowQ2fjXO/TpQwqidtItpg3DgSwfWLPqp',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': '6+OMon/6TFfsFnafECM70JJOupE62RQmsFe4fVlZ4svmZtt/QoYbcTVXcCiDkuWyowQ2fjXO/TpQwqidtItpg3DgSwfWLPqp',
   'x-amz-request-id': '3QNBEXEA1NT4MQ5J',
   'server': 'AmazonS3',
   'date': 'Sun, 06 Sep 2026 20:30:15 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"16c76dd84c9f66290eb173fe8215be16"',
   'x-amz-checksum-crc32': 'QhPhfw==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0'},
  'RetryAttempts': 0},
 'ETag': '"16c76dd84c9f66290eb173fe8215be16"',
 'ChecksumCRC32': 'QhPhfw==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

This code is for initial load of Company profile data

In [ ]:
s3 = boto3.client('s3')
profile_url = 'https://financialmodelingprep.com/stable/profile?symbol='
timeout = (3.05, 30)
filename = "company_profile_raw_" + datetime.now(UTC).strftime("%Y%m%d_%H%M%S") + ".json"
bucket = 'phile-findata1-raw-data'
key_path = 'to_process/'

def extract_company_profiles(profile_url: str, timeout:tuple) -> list[dict]:
    all_records = []
    config_bucket = 'phile-findata1-configdata'
    key_file = 'fmp87.json'

    config_response = s3.get_object(Bucket=config_bucket, Key=key_file)
    config_content = config_response['Body'].read().decode('utf-8')
    json_config = json.loads(config_content)

    for ticker in json_config['fmp87']:
        response = requests.get(
            profile_url + ticker,
            params={'apikey': settings.apikey},
            timeout=timeout
        )
        response.raise_for_status()
        data = response.json()
        all_records.extend(data)
       
    return all_records

profiles = extract_company_profiles(profile_url, timeout)

s3.put_object(
    Bucket=bucket,
    Key=key_path + filename,
    Body=json.dumps(profiles),
    ContentType='application/json'
)

{'ResponseMetadata': {'RequestId': 'Q10Y0AQM1BDB818Y',
  'HostId': 'Sl/mhSHCxvcwnbF/hEcntxtB3n5VjS2x/9vCvLLUpDsqXncBA8rK4fa/P2jrkzd7TWhUNdMJCIbfrRGgrtpbaHhqTWrS0/0B',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'Sl/mhSHCxvcwnbF/hEcntxtB3n5VjS2x/9vCvLLUpDsqXncBA8rK4fa/P2jrkzd7TWhUNdMJCIbfrRGgrtpbaHhqTWrS0/0B',
   'x-amz-request-id': 'Q10Y0AQM1BDB818Y',
   'server': 'AmazonS3',
   'date': 'Sun, 06 Sep 2026 20:47:24 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"ae4fbeef3c05469af9bcfa2a7b2add73"',
   'x-amz-checksum-crc32': '2LShzA==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0'},
  'RetryAttempts': 0},
 'ETag': '"ae4fbeef3c05469af9bcfa2a7b2add73"',
 'ChecksumCRC32': '2LShzA==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

Code for daily pull of yesterday stock prices

In [ ]:
import json
from datetime import UTC, datetime, timedelta

import boto3

s3 = boto3.client('s3')
api_url = 'https://financialmodelingprep.com/stable/historical-price-eod/full?symbol='
timeout = (3.05, 30)
filename = "stock_prices_raw_" + datetime.now(UTC).strftime("%Y%m%d_%H%M%S") + ".json"
bucket = 'phile-findata1-raw-data'
key_path = 'to_process/'

def extract_eod_stockprices(api_url: str, timeout:tuple) -> list[dict]:
    """Extract new stock prices from the FMP api"""
    all_records = []
    config_bucket = 'phile-findata1-configdata'
    key_file = 'dow30_test.json'
    yesterday = datetime.now(UTC) - timedelta(days=1)
    yesterday_str = yesterday.strftime('%Y-%m-%d')

    #Loads config file from S3 which has the 87 free tier companies in it
    config_response = s3.get_object(Bucket=config_bucket, Key=key_file)
    config_content = config_response['Body'].read().decode('utf-8')
    json_config = json.loads(config_content)

    for ticker in json_config['dow30']:
        response = requests.get(
            api_url + ticker,
            params={'from': yesterday_str, 'to': yesterday_str, 'apikey': settings.apikey},
            timeout=timeout
        )

        response.raise_for_status()
        data = response.json()
        all_records.extend(data)
       
    return all_records
    

stock_prices = extract_eod_stockprices(api_url, timeout)

s3.put_object(
    Bucket=bucket,
    Key=key_path + filename,
    Body=json.dumps(stock_prices),
    ContentType='application/json'
)


{'ResponseMetadata': {'RequestId': 'TJS1Q35C83TEK7P9',
  'HostId': 'yoYqkkA2VXkSUTNsUjQAGt4aBTSdBVISRxA2m9YBbzqC1XzJn1KfyRH/5jeTQZSWbog5tbLNR8Xxw4FH43ibucU2+0cmM1pi',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'yoYqkkA2VXkSUTNsUjQAGt4aBTSdBVISRxA2m9YBbzqC1XzJn1KfyRH/5jeTQZSWbog5tbLNR8Xxw4FH43ibucU2+0cmM1pi',
   'x-amz-request-id': 'TJS1Q35C83TEK7P9',
   'server': 'AmazonS3',
   'date': 'Sun, 06 Sep 2026 20:55:32 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"d751713988987e9331980363e24189ce"',
   'x-amz-checksum-crc32': 'DUy7KQ==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0'},
  'RetryAttempts': 0},
 'ETag': '"d751713988987e9331980363e24189ce"',
 'ChecksumCRC32': 'DUy7KQ==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}